# Built IN

In [3]:
!pip install langchain langchain-core langchain-community pydantic duckduckgo-search langchain_experimental -U ddgs

In [4]:
from langchain_community.tools import DuckDuckGoSearchRun
search_tool = DuckDuckGoSearchRun()
results = search_tool.invoke('top news in india today')
print(results)

2 days ago - India News | Latest India News | Read latest and breaking news from India . Today ' s top India news headlines, news on Indian politics, elections, government, business, technology, and Bollywood. 19 hours ago - Latest news headlines from India & around the world. Check out today ’ s news coverage live with videos & photos on NDTV.com. 11 hours ago - Latest News India : Get the breaking news from India on indian government, economic developments, entertainment, technology, lifestyle & more. Find comprehensive updates and insights from Business Standard. 19 hours ago - Get all the latest news headlines from India and around the world. Explore more for today ’ s news coverage with videos & photos on Business Standard. Business News : Get latest news on share market, personal finance news , economy news , stock market updates, company news , India Pakistan conflict news , politics news , Assembly Election Results 2024, breaking news at Business Standard. Catch all the latest 

In [5]:
print(search_tool.name)
print(search_tool.description)
print(search_tool.args)

duckduckgo_search
A wrapper around DuckDuckGo Search. Useful for when you need to answer questions about current events. Input should be a search query.
{'query': {'description': 'search query to look up', 'title': 'Query', 'type': 'string'}}


In [48]:
from langchain_community.tools import ShellTool

shell_tool = ShellTool()

results = shell_tool.invoke('sudo apt-get update ')

print(results)

Executing command:
 sudo apt-get update 
sudo: a terminal is required to read the password; either use the -S option to read from standard input or configure an askpass helper
sudo: a password is required



/opt/homebrew/Caskroom/miniforge/base/envs/tf_numpy2/lib/python3.11/site-packages/langchain_community/tools/shell/tool.py:33: UserWarning: The shell tool has no safeguards by default. Use at your own risk.
  warnings.warn(


## Custom Tools


In [49]:
from langchain_core.tools import tool

In [50]:
@tool
def multiply(a: int, b:int) -> int:
    """Multiply two given numbers"""
    return a*b

In [51]:
result = multiply.invoke({"a":3, "b":5})

In [26]:
print(result)

15


In [27]:
print(multiply.name)
print(multiply.description)
print(multiply.args)

multiply
Multiply two given numbers
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [19]:
print(multiply.args_schema.model_json_schema())

{'description': 'Multiply two numbers', 'properties': {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'title': 'multiply', 'type': 'object'}


## using Structured Tool


In [52]:
from langchain.tools import StructuredTool
from pydantic import BaseModel, Field

In [53]:
class MultiplyInput(BaseModel):
    a: int = Field(required=True, description="The first number to add")
    b: int = Field(required=True, description="The second number to add")

In [54]:
def multiply_func(a: int, b: int) -> int:
    return a * b

In [55]:
multiply_tool = StructuredTool.from_function(
    func=multiply_func,
    name="multiply",
    description="Multiply two numbers",
    args_schema=MultiplyInput
)

In [35]:
result = multiply_tool.invoke({'a':3, 'b':3})

print(result)
print(multiply_tool.name)
print(multiply_tool.description)
print(multiply_tool.args)

9
multiply
Multiply two numbers
{'a': {'description': 'The first number to add', 'required': True, 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number to add', 'required': True, 'title': 'B', 'type': 'integer'}}


## Using Base Tool

In [56]:
from langchain.tools import BaseTool
from typing import Type

In [57]:
class MultiplyInput(BaseModel):
    a: int = Field(required=True, description="The first number to add")
    b: int = Field(required=True, description="The second number to add")

In [59]:
class MultiplyTool(BaseTool):
    """Multiplies the number"""
    name: str = "multiply"
    description: str = "Multiply two numbers"

    args_schema: Type[BaseModel] = MultiplyInput

    def _run(self, a: int, b: int) -> int:
        """Multiply the two numbers"""
        return a * b

In [60]:
multiply_tool = MultiplyTool()

In [61]:
result = multiply_tool.invoke({'a':3, 'b':3})

print(result)
print(multiply_tool.name)
print(multiply_tool.description)

print(multiply_tool.args)

9
multiply
Multiply two numbers
{'a': {'description': 'The first number to add', 'required': True, 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number to add', 'required': True, 'title': 'B', 'type': 'integer'}}


## Toolkit

In [63]:
from langchain_core.tools import tool

# Custom tools
@tool
def add(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b

In [64]:
class MathToolkit:
    def get_tools(self):
        return [add, multiply]

In [65]:
toolkit = MathToolkit()
tools = toolkit.get_tools()

for tool in tools:
    print(tool.name, "=>", tool.description)

add => Add two numbers
multiply => Multiply two numbers
